# Análisis del comportamiento de una aplicación bajo diferentes escenarios de despliegue

**Curso:** Principios y prácticas de Desarrollo de Software Orientado a Objetos
**Profesor:** Gabriel Rodrigo Pedraza Ferreira
**Integrantes:** **[COMPLETAR]**
**Repositorio:** **[COMPLETAR: enlace]**

El notebook descarga los resultados de JMeter desde el repositorio y genera todas las tablas y gráficas. En Google Colab: *Entorno de ejecución → Ejecutar todas*.

## 1. Introducción y objetivos

Se desplegó una misma aplicación de varias formas y se midió cómo responde a distintas cargas:

| Escenario | Herramienta | Réplicas de la API |
|---|---|---|
| Una sola máquina | Docker Compose | 1 |
| Clúster de 1 nodo | Kubernetes (k3s) | 1, 2 y 3 |
| Clúster de 2 nodos | Kubernetes (k3s) | 1, 2 y 3 |
| Clúster de 3 nodos | Kubernetes (k3s) | 1, 2 y 3 |

En cada configuración se generó carga con JMeter (10, 25, 50 y 100 usuarios concurrentes) y se midieron el **tiempo medio de respuesta** y el **throughput** (peticiones atendidas por segundo).

**Objetivo general:** analizar cómo el entorno de despliegue, el número de réplicas y el número de nodos afectan el rendimiento de la aplicación, en especial el escalado horizontal.

**Objetivos específicos:**

- Desplegar la aplicación con Docker Compose y en Kubernetes con 1, 2 y 3 nodos.
- Medir con JMeter el tiempo medio de respuesta y el throughput con varias cargas y números de réplicas.
- Registrar el consumo de CPU y memoria de nodos y pods durante las pruebas.
- Comparar los escenarios, encontrar tendencias y cuellos de botella, y sacar conclusiones y recomendaciones.

## 2. Descripción de la aplicación

Es la API REST de una librería, hecha para este proyecto. Se escribió en **Go** con la librería estándar y el driver de PostgreSQL `lib/pq`, porque produce una imagen de unos 8 MB, arranca en menos de un segundo y usa alrededor de 10 MB de memoria por réplica. Así cada réplica nueva está lista casi de inmediato y las máquinas virtuales gastan sus recursos en atender peticiones.

**Base de datos:** PostgreSQL 16. El archivo `db/init.sql` crea las tablas y genera los datos con una semilla fija, así todos los escenarios parten de la misma base:

| Tabla | Registros iniciales | Contenido |
|---|---|---|
| `categorias` | 12 | Novela, Ciencia, Programación... |
| `autores` | 1.000 | Nombre y país |
| `libros` | 10.000 | ISBN, título, autor, categoría, año, precio y stock |
| `pedidos` | 0 | Se llena durante las pruebas |

**Endpoints:**

| Método | Ruta | Qué hace |
|---|---|---|
| GET | `/api/libros/{id}` | Un libro con su autor y categoría (consulta con dos JOIN) |
| GET | `/api/libros?pagina=1&tamano=20&categoria=3` | Listado paginado, con filtro opcional por categoría |
| GET | `/api/libros/buscar?q=jardin` | Búsqueda por título o autor |
| GET | `/api/categorias/estadisticas` | Libros, precio promedio y unidades por categoría |
| POST | `/api/pedidos` | Crea un pedido: valida la cantidad, descuenta el inventario y registra el pedido en una transacción; 10 % de descuento desde 3 unidades; responde 409 si no hay unidades |
| GET | `/api/pedidos/{id}` | Consulta un pedido |
| GET | `/salud` y `/listo` | Usados por Docker y Kubernetes para saber si la réplica está viva y lista |

Cada respuesta lleva la cabecera `X-Replica` con el nombre del pod que la atendió, lo que permite comprobar el balanceo entre réplicas.

Recorrido de una petición en Kubernetes:

```
JMeter (VM jmeter) ──HTTP──▶ NodePort 30080 en vm1 ──▶ pod de la API (cualquier nodo) ──SQL──▶ PostgreSQL (vm1)
```

In [ ]:
import os

REPO_URL = "https://github.com/USUARIO/REPOSITORIO.git"   # [COMPLETAR]

# En Colab se descarga el repositorio; si el notebook se abre dentro del repositorio, se usa tal cual
if not os.path.exists("resultados"):
    !git clone -q {REPO_URL} repo
    %cd repo

In [ ]:
import glob
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 1)

def mostrar(ruta, desde=None, hasta=None):
    lineas = open(ruta, encoding="utf-8").read().splitlines()
    print(f"# ---- {ruta} ----")
    print("\n".join(lineas[desde:hasta]))

mostrar("db/init.sql")

## 3. Infraestructura

Todo corre en un solo computador con **VirtualBox** y **Vagrant** (archivo `Vagrantfile`). Las VM usan Ubuntu 24.04.

| Máquina | IP | CPU | RAM | Uso |
|---|---|---|---|---|
| vm1 | 192.168.56.11 | 2 | 4 GB | Docker Compose (fase 1); servidor de Kubernetes; siempre aloja PostgreSQL |
| vm2 | 192.168.56.12 | 2 | 3 GB | Segundo nodo (fases 3 y 4) |
| vm3 | 192.168.56.13 | 2 | 3 GB | Tercer nodo (fase 4) |
| jmeter | 192.168.56.20 | 2 | 2 GB | Genera la carga con JMeter 5.6.3 |

**Computador físico:** Intel Core i5-10300H (4 núcleos, 8 hilos), 16 GB de RAM, Windows. **[COMPLETAR: versión de Windows]**

**Kubernetes:** k3s, una distribución liviana que trae todo en un solo binario. Se desactivaron Traefik y el balanceador de servicio porque no se usan. La red entre nodos es Flannel sobre la red privada 192.168.56.x. `metrics-server` viene incluido y permite medir CPU y memoria con `kubectl top`.

Las cuatro VM comparten el mismo procesador físico: con todo encendido suman 8 CPU virtuales, tantas como hilos tiene el procesador. Esto se tiene en cuenta al analizar el efecto de agregar nodos.

## 4. Configuración de despliegue de cada escenario

### 4.1 Fase 1: Docker Compose

vm1 corre dos contenedores, PostgreSQL y la API. Los límites son los mismos que en Kubernetes: 1 CPU y 512 MB para la base, 0,5 CPU y 128 MB para la API. La base guarda sus datos en memoria (`tmpfs`), así cada despliegue parte de los mismos 10.000 libros.

```bash
vagrant ssh vm1 -c "bash /vagrant/scripts/modo-compose.sh"
```

In [ ]:
mostrar("api/Dockerfile")
mostrar("docker/docker-compose.yml")

### 4.2 Fases 2, 3 y 4: Kubernetes

| Archivo | Contenido |
|---|---|
| `k8s/00-namespace.yaml` | Namespace `libreria` |
| `k8s/01-db-init.yaml` | ConfigMap con `init.sql` |
| `k8s/02-db.yaml` | PostgreSQL: 1 réplica fija en vm1 y su Service |
| `k8s/03-api.yaml` | API: Deployment escalable y Service NodePort 30080 |

Decisiones:

- **La base no se escala.** Tiene una réplica y está fija en vm1 (`nodeSelector`). Solo se escala la API.
- **Las réplicas se reparten entre nodos** con `topologySpreadConstraints` (máximo 1 réplica de diferencia entre nodos).
- **Límites por réplica de la API:** 0,5 CPU y 128 MiB. `GOMAXPROCS=1` para que Go no intente usar más procesador del que tiene asignado.
- **Probes:** `readinessProbe` en `/listo` (comprueba la base) y `livenessProbe` en `/salud`.
- **Imagen:** se construye en vm1 durante la fase 1 y se importa en cada nodo desde `build/libreria-api.tar`, sin pasar por Docker Hub.

Despliegue y escalado:

```bash
vagrant ssh vm1 -c "bash /vagrant/scripts/modo-k8s.sh"
vagrant ssh jmeter -c "bash /vagrant/pruebas/serie-k8s.sh k8s1n"   # 1, 2 y 3 réplicas
vagrant up vm2      # se une sola al clúster
vagrant ssh jmeter -c "bash /vagrant/pruebas/serie-k8s.sh k8s2n"
vagrant up vm3
vagrant ssh jmeter -c "bash /vagrant/pruebas/serie-k8s.sh k8s3n"
```

`serie-k8s.sh` baja la API a 0 réplicas y la sube a N antes de cada configuración, para que los pods se repartan desde cero.

In [ ]:
mostrar("k8s/02-db.yaml")
mostrar("k8s/03-api.yaml")

### 4.3 Ubicación de los pods

Guardada automáticamente antes de cada configuración en `resultados/ubicacion/`.

In [ ]:
ubicacion = []
for ruta in sorted(glob.glob("resultados/ubicacion/*.txt")):
    esc, rep = re.match(r"(k8s\dn)_(\d)rep", os.path.basename(ruta)).groups()
    for linea in open(ruta).read().splitlines()[1:]:
        partes = linea.split()
        nodo = next((x for x in partes if re.fullmatch(r"vm\d", x)), None)
        if nodo:
            ubicacion.append({"escenario": esc, "replicas": int(rep),
                              "pod": "db" if partes[0].startswith("db-") else "api", "nodo": nodo})
ubicacion = pd.DataFrame(ubicacion)
if len(ubicacion):
    tabla_ubic = (ubicacion.groupby(["escenario", "replicas", "pod"]).nodo
                  .apply(lambda s: ", ".join(sorted(s))).unstack("pod"))
    display(tabla_ubic)
else:
    print("No hay archivos de ubicacion")

## 5. Monitorización

Durante cada configuración de Kubernetes, `serie-k8s.sh` guardó cada 10 segundos la salida de `kubectl top nodes` y `kubectl top pods` (datos de `metrics-server`) en `resultados/monitoreo/`. Con eso se calcula el consumo promedio de CPU de cada nodo, de la base y de las réplicas de la API mientras corrían las pruebas.

In [ ]:
def a_milicores(v):
    return float(v[:-1]) if v.endswith("m") else float(v) * 1000

registros = []
for ruta in sorted(glob.glob("resultados/monitoreo/*.txt")):
    esc, rep = re.match(r"(k8s\dn)_(\d)rep", os.path.basename(ruta)).groups()
    for linea in open(ruta).read().splitlines():
        p = linea.split()
        if len(p) >= 5 and p[0].startswith("vm") and p[2].endswith("%"):
            registros.append({"escenario": esc, "replicas": int(rep), "componente": p[0],
                              "cpu_m": a_milicores(p[1])})
        elif len(p) == 3 and (p[0].startswith("api-") or p[0].startswith("db-")) and p[1].endswith("m"):
            registros.append({"escenario": esc, "replicas": int(rep),
                              "componente": "API (todas las réplicas)" if p[0].startswith("api-") else "PostgreSQL",
                              "cpu_m": a_milicores(p[1]), "muestra": linea})
monitoreo = pd.DataFrame(registros)
if len(monitoreo):
    # Para la API se suman las replicas de cada muestra; luego se promedia en el tiempo
    cpu = (monitoreo.groupby(["escenario", "replicas", "componente"]).cpu_m.mean()
           .unstack("componente"))
    if "API (todas las réplicas)" in cpu:
        cpu["API (todas las réplicas)"] = cpu["API (todas las réplicas)"] * cpu.index.get_level_values("replicas")
    print("CPU promedio durante las pruebas (milicores; 1000 = 1 núcleo)")
    display(cpu.round(0))
else:
    print("No hay archivos de monitoreo")

In [ ]:
if len(monitoreo):
    comp = cpu[[c for c in ["PostgreSQL", "API (todas las réplicas)"] if c in cpu]].reset_index()
    comp["config"] = comp.escenario.str.replace("k8s", "").str.replace("n", " nodo(s)") + " / " + comp.replicas.astype(str) + "r"
    comp.set_index("config")[[c for c in ["PostgreSQL", "API (todas las réplicas)"] if c in comp]].plot(
        kind="bar", figsize=(11, 3.8), color=["#762a83", "#5aae61"])
    plt.ylabel("CPU (milicores)")
    plt.xlabel("")
    plt.title("Consumo de CPU durante las pruebas")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Metodología de las pruebas con JMeter

**Plan** (`jmeter/plan.jmx`). Cada usuario virtual repite sin pausa:

1. `GET /api/libros/{id}` con un id al azar entre 1 y 10.000
2. `GET /api/libros?pagina={1..500}&tamano=20`
3. `GET /api/libros/{id}` con otro id al azar
4. `POST /api/pedidos` con un libro al azar y una cantidad de 1 a 4

Es decir, 75 % lecturas y 25 % escrituras, sobre datos distintos en cada petición.

**Ejecución** (`pruebas/correr.sh`), igual en todas las configuraciones:

| Parámetro | Valor |
|---|---|
| Usuarios concurrentes | 10, 25, 50 y 100 |
| Subida de usuarios | 10 s (se descartan en el análisis) |
| Duración de cada prueba | 60 s |
| Repeticiones | **[COMPLETAR: 1 o 2]** |
| Pausa entre pruebas | 10 s |
| Calentamiento | 20 s con 10 usuarios antes de cada configuración, sin guardar |
| Modo | consola (`jmeter -n`) en una VM propia, distinta a la de la aplicación |

Se eligieron pruebas cortas para poder cubrir las 10 configuraciones en una sola sesión, con las mismas condiciones para todas.

**Procesamiento:** cada prueba deja un CSV con una fila por petición (`escenario_Nrep_Uu_rR.csv`). Se descartan los 10 s de subida y se calculan el throughput (peticiones / segundos medidos), el tiempo medio, el percentil 95 y el porcentaje de errores. Luego se promedian las repeticiones.

In [ ]:
RAMPUP_S = 10
DURACION_S = 60
PATRON = re.compile(r"(?P<escenario>compose|k8s[123]n)_(?P<replicas>\d)rep_(?P<usuarios>\d+)u_r(?P<corrida>\d+)")
NOMBRES = {"compose": "Docker Compose", "k8s1n": "K8s 1 nodo", "k8s2n": "K8s 2 nodos", "k8s3n": "K8s 3 nodos"}
ORDEN = list(NOMBRES.values())

def metricas(df, segundos):
    return {"peticiones": len(df), "throughput": len(df) / segundos, "t_medio": df.elapsed.mean(),
            "p95": df.elapsed.quantile(0.95),
            "errores_pct": 100 * (df.success.astype(str).str.lower() != "true").mean()}

filas, por_endpoint, series, incompletas = [], [], {}, []
for ruta in sorted(glob.glob("resultados/*.csv")):
    nombre = os.path.basename(ruta)
    m = PATRON.match(nombre)
    if not m:
        continue
    df = pd.read_csv(ruta, low_memory=False)
    dur = (df.timeStamp.max() - df.timeStamp.min()) / 1000
    if dur < DURACION_S - 5:
        incompletas.append({"archivo": nombre, "duracion_s": round(dur)})
        continue
    inicio = df.timeStamp.min()
    series[nombre] = df.groupby((df.timeStamp - inicio) // 2000).size() / 2
    df = df[df.timeStamp >= inicio + RAMPUP_S * 1000]
    segundos = ((df.timeStamp + df.elapsed).max() - df.timeStamp.min()) / 1000
    filas.append({**m.groupdict(), **metricas(df, segundos)})
    for etiqueta, g in df.groupby("label"):
        por_endpoint.append({**m.groupdict(), "endpoint": etiqueta, **metricas(g, segundos)})

def preparar(t):
    t = pd.DataFrame(t)
    for c in ("replicas", "usuarios", "corrida"):
        t[c] = t[c].astype(int)
    t["escenario"] = pd.Categorical(t.escenario.map(NOMBRES), ORDEN, ordered=True)
    return t

corridas = preparar(filas)
endpoints = preparar(por_endpoint)
print(f"Pruebas válidas: {len(corridas)}   Peticiones analizadas: {corridas.peticiones.sum():,.0f}")
print(f"Pruebas con errores: {(corridas.errores_pct > 0).sum()}")
print("Pruebas incompletas (no se usan):", incompletas if incompletas else "ninguna")
display(corridas.pivot_table(index=["escenario", "replicas"], columns="usuarios",
                             values="corrida", aggfunc="count", observed=True))

## 7. Resultados

### 7.1 Tablas resumen

In [ ]:
resumen = (corridas.groupby(["escenario", "replicas", "usuarios"], observed=True)
           [["throughput", "t_medio", "p95", "errores_pct"]].mean())
print("Throughput (peticiones por segundo)")
display(resumen.throughput.unstack("usuarios").round(1))
print("Tiempo medio de respuesta (ms)")
display(resumen.t_medio.unstack("usuarios").round(1))
print("Percentil 95 (ms)")
display(resumen.p95.unstack("usuarios").round(0))
print("Errores (%)")
display(resumen.errores_pct.unstack("usuarios").round(2))

### 7.2 Efecto del número de réplicas

In [ ]:
k8s = corridas[corridas.escenario != "Docker Compose"].copy()
k8s["escenario"] = k8s.escenario.cat.remove_unused_categories()
COLOR_REPLICAS = {1: "#c2a5cf", 2: "#9970ab", 3: "#40004b"}
USUARIOS = sorted(corridas.usuarios.unique())

for metrica, titulo in [("throughput", "Throughput (peticiones/s)"), ("t_medio", "Tiempo medio (ms)")]:
    g = sns.relplot(data=k8s, x="usuarios", y=metrica, hue="replicas", col="escenario", kind="line",
                    marker="o", errorbar="sd", palette=COLOR_REPLICAS, height=3.6, aspect=1.05)
    g.set_axis_labels("Usuarios concurrentes", titulo)
    g.set_titles("{col_name}")
    g.legend.set_title("Réplicas")
    g.set(xticks=USUARIOS)
    if metrica == "throughput":
        g.set(ylim=(0, None))
    g.figure.suptitle(f"{titulo} según el número de réplicas", y=1.04)
    plt.show()

### 7.3 Efecto del número de nodos

In [ ]:
COLOR_NODOS = {"K8s 1 nodo": "#1b7837", "K8s 2 nodos": "#c51b7d", "K8s 3 nodos": "#542788"}
for metrica, titulo in [("throughput", "Throughput (peticiones/s)"), ("t_medio", "Tiempo medio (ms)")]:
    g = sns.relplot(data=k8s, x="usuarios", y=metrica, hue="escenario", col="replicas", kind="line",
                    marker="o", errorbar="sd", palette=COLOR_NODOS, height=3.6, aspect=1.05)
    g.set_axis_labels("Usuarios concurrentes", titulo)
    g.set_titles("{col_name} réplica(s)")
    g.legend.set_title("Escenario")
    g.set(xticks=USUARIOS)
    if metrica == "throughput":
        g.set(ylim=(0, None))
    g.figure.suptitle(f"{titulo} según el número de nodos", y=1.04)
    plt.show()

### 7.4 Las nueve configuraciones de Kubernetes con la carga más alta

In [ ]:
CARGA = max(USUARIOS)
alta = resumen.xs(CARGA, level="usuarios").drop("Docker Compose", level="escenario", errors="ignore")
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.heatmap(alta.throughput.unstack("replicas"), annot=True, fmt=".0f", cmap="PuRd",
            cbar_kws={"label": "peticiones/s"}, ax=ejes[0])
ejes[0].set_title(f"Throughput con {CARGA} usuarios")
sns.heatmap(alta.t_medio.unstack("replicas"), annot=True, fmt=".0f", cmap="PuRd",
            cbar_kws={"label": "ms"}, ax=ejes[1])
ejes[1].set_title(f"Tiempo medio con {CARGA} usuarios")
for ax in ejes:
    ax.set_xlabel("Réplicas"); ax.set_ylabel("")
plt.tight_layout()
plt.show()

base = alta.loc[("K8s 1 nodo", 1), "throughput"]
print(f"Cambio del throughput frente a K8s 1 nodo con 1 réplica (%), {CARGA} usuarios")
display((100 * (alta.throughput / base - 1)).unstack("replicas").round(1))

### 7.5 Docker Compose frente a Kubernetes (1 réplica)

In [ ]:
una = corridas[corridas.replicas == 1]
g = sns.relplot(data=una, x="usuarios", y="throughput", hue="escenario", kind="line", marker="o",
                errorbar="sd", palette={"Docker Compose": "#878787", **COLOR_NODOS}, height=3.6, aspect=1.6)
g.set_axis_labels("Usuarios concurrentes", "Throughput (peticiones/s)")
g.set(xticks=USUARIOS, ylim=(0, None))
g.figure.suptitle("Docker Compose frente a Kubernetes, 1 réplica", y=1.04)
plt.show()
display(una.groupby(["escenario", "usuarios"], observed=True)[["throughput", "t_medio"]].mean()
        .unstack("usuarios").round(1))

### 7.6 Tiempo por tipo de petición

In [ ]:
e = endpoints[(endpoints.usuarios == CARGA) & (endpoints.escenario != "Docker Compose")].copy()
e["config"] = e.escenario.astype(str).str.replace("K8s ", "") + " / " + e.replicas.astype(str) + "r"
orden = [f"{n} / {r}r" for n in ["1 nodo", "2 nodos", "3 nodos"] for r in (1, 2, 3)]
plt.figure(figsize=(11, 3.8))
sns.barplot(data=e, x="config", y="t_medio", hue="endpoint", order=[o for o in orden if o in set(e.config)],
            palette=["#762a83", "#5aae61", "#c51b7d"], errorbar=None)
plt.xlabel("Escenario / réplicas"); plt.ylabel("Tiempo medio (ms)")
plt.title(f"Tiempo medio por tipo de petición con {CARGA} usuarios")
plt.xticks(rotation=30); plt.legend(title=""); plt.tight_layout(); plt.show()

### 7.7 Estabilidad durante una prueba y ley de Little

Throughput cada 2 segundos en una prueba de la carga más alta, para la mejor y la peor configuración. En una prueba sin pausas se cumple aproximadamente **usuarios ≈ throughput × tiempo medio**: si la tabla da valores cercanos al número de usuarios, el aumento del tiempo de respuesta es tiempo de espera en cola.

In [ ]:
mejor = alta.throughput.idxmax(); peor = alta.throughput.idxmin()
plt.figure(figsize=(10, 3.4))
for (esc, rep), color in [(mejor, "#1b7837"), (peor, "#542788")]:
    clave = {v: k for k, v in NOMBRES.items()}[esc]
    archivo = f"{clave}_{rep}rep_{CARGA}u_r1.csv"
    if archivo in series:
        s = series[archivo]
        plt.plot(s.index * 2, s.values, label=f"{esc}, {rep} réplica(s)", color=color)
plt.axvline(RAMPUP_S, color="gray", linestyle="--")
plt.xlabel("Segundos desde el inicio"); plt.ylabel("peticiones/s"); plt.ylim(0, None)
plt.title(f"Throughput durante una prueba de {CARGA} usuarios"); plt.legend(); plt.tight_layout(); plt.show()

little = resumen.reset_index()
little["usuarios_estimados"] = little.throughput * little.t_medio / 1000
display(little.pivot_table(index=["escenario", "replicas"], columns="usuarios",
                           values="usuarios_estimados", observed=True).round(0))

## 8. Análisis comparativo

**[COMPLETAR con los resultados.]** Puntos a revisar en las gráficas y tablas:

- **Réplicas en 1 nodo:** ¿sube el throughput al pasar de 1 a 2 réplicas? ¿Y de 2 a 3? En vm1 hay 2 CPU para la base y las réplicas, así que la mejora debería frenarse cuando se ocupan.
- **Réplicas en 2 y 3 nodos:** con más nodos, las réplicas tienen CPU propia. ¿Escala mejor que en 1 nodo?
- **Nodos con 1 réplica:** ¿cambia algo si la única réplica queda en otro nodo distinto a la base? (costo de la red entre nodos).
- **Cuello de botella:** con la tabla de CPU de la sección 5, ver si la base o las réplicas llegan a su límite (1000m y 500m por réplica).
- **Docker Compose frente a K8s 1 nodo:** ¿Kubernetes agrega algún costo con 1 réplica?
- **Errores y estabilidad.**
- **Límite del hardware:** las 4 VM comparten 4 núcleos físicos.

## 9. Conclusiones

**[COMPLETAR]**

## 10. Recomendaciones

**[COMPLETAR]**

## 11. Limitaciones

- Las cuatro VM, incluida la de JMeter, corren en el mismo computador (4 núcleos físicos). Los nodos nuevos no agregan procesador real y JMeter compite con la aplicación.
- Las pruebas son cortas (60 s) **[y con una sola repetición]** para poder hacer todas las configuraciones en una sesión.